##import


In [3]:
import numpy as np
import pandas as pd
import os
import torch
import torch.nn.functional as F
import torch.nn as nn
from tqdm import tqdm
import random
import math
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow import keras
import tensorflow as tf
from tensorflow.keras.layers import InputLayer, Reshape, Conv1D, BatchNormalization, DepthwiseConv1D, MaxPool1D, GlobalAvgPool1D, Dropout, Dense
from tensorflow.keras import layers, optimizers
import torch.optim as optim

In [4]:
PREPROCESS = True
UNITS = 256 # Transformer

# Transformer
NUM_BLOCKS = 4
MLP_RATIO = 2

# Dropout
EMBEDDING_DROPOUT = 0.00
MLP_DROPOUT_RATIO = 0.30
CLASSIFIER_DROPOUT_RATIO = 0.10

# Initiailizers
INIT_HE_UNIFORM = tf.keras.initializers.he_uniform
INIT_GLOROT_UNIFORM = tf.keras.initializers.glorot_uniform
INIT_ZEROS = tf.keras.initializers.constant(0.0)

# Activations
GELU = tf.keras.activations.gelu

# N_EPOCHS = 150
# LR_MAX = 1e-4
# N_WARMUP_EPOCHS = 0
# WD_RATIO = 0.05
# NUM_CLASSES = 250

In [5]:
ROWS_PER_FRAME = 80
def load_relevant_data_subset(pq_path):
    if not os.path.exists(pq_path):
        raise FileNotFoundError(f"File not found: {pq_path}")
    if os.path.getsize(pq_path) == 0:
        raise ValueError(f"Parquet file is empty: {pq_path}")
    data_columns = ['x', 'y', 'z']
    data = pd.read_parquet(pq_path, columns=data_columns)
    # Convert NaN values to 0
    data = data.fillna(0)
    n_frames = int(len(data) / ROWS_PER_FRAME)
    data = data.values.reshape(n_frames, ROWS_PER_FRAME, len(data_columns))
    return data.astype(np.float32)
    # Check if the function returns in required dimensions
single_pq = load_relevant_data_subset(r'/kaggle/input/big-dataset/subsample_paraquet/-02o_0vVwzI--0.parquet')
print(single_pq.shape)

(64, 80, 3)


##Mapping


In [ ]:
IDX_MAP = {
    "mediapipe": np.array([
        # window 1: face_head
        38,36,34,31,32,37,0,8,40,39,42,44,46,41,43,45,
        
        # window 2: left_hand
        48,79,77,76,75,73,72,71,69,68,67,66,64,63,61,60,
    
        # window 3: left hand_with_arms
        42,44,46,52,76,78,79,72,75,68,71,64,67,60,52,63,

        # window 4: right_hand_shape
        41,43,45,51,26,28,29,22,25,18,21,14,17,10,12,13,

        # window 5: dynamic_detail
        49,29,27,26,25,23,22,21,19,18,17,15,14,13,11,10,
    ])
}

In [11]:
IDX_MAP["mediapipe"].shape

(80,)

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class FeaturePreprocess(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x_in):
        n_frames = x_in.shape[0]

        selected_landmarks = x_in[:, IDX_MAP["mediapipe"]]  

        selected_landmarks[torch.isnan(selected_landmarks)] = 0

        return selected_landmarks


In [13]:
x_in = torch.tensor(load_relevant_data_subset('/kaggle/input/big-dataset/subsample_paraquet/-02o_0vVwzI--0.parquet'))
feature_preprocess = FeaturePreprocess()
print(feature_preprocess(x_in).shape, x_in[0])

torch.Size([64, 80, 3]) tensor([[ 5.0071e-01,  3.4195e-01, -2.1444e-02],
        [ 5.0077e-01,  3.2164e-01, -2.4070e-02],
        [ 5.0144e-01,  2.9013e-01, -4.9109e-02],
        [ 5.0064e-01,  3.4609e-01, -2.0002e-02],
        [ 5.0069e-01,  3.4877e-01, -1.6864e-02],
        [ 5.0084e-01,  3.4951e-01, -1.2875e-02],
        [ 5.0138e-01,  3.5169e-01, -1.1275e-02],
        [ 5.0129e-01,  3.5473e-01, -1.2719e-02],
        [ 5.0117e-01,  3.5895e-01, -1.5092e-02],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+0

In [14]:
sampled_df=pd.read_csv(r"/kaggle/input/big-dataset-csv/iSign_v1.1.csv")
uids = sampled_df['uid'].tolist()
texts = sampled_df['text'].tolist()


In [15]:
signs = []
paths = []
for uid, text in zip(uids, texts):
    phrase=text
    path=f"/kaggle/input/big-dataset/subsample_paraquet/{uid}.parquet"
    signs.append(phrase)
    paths.append(path)

data = {
    'sign': signs,
    'path': paths,
}
train_df = pd.DataFrame(data)

csv_filename = 'train.csv'
train_df.to_csv(csv_filename, index=False)

In [16]:
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer()
tokenizer.train(files=[r"/kaggle/input/outputsid/output.txt"], vocab_size=45000, special_tokens=["<s>", "</s>", "<pad>", "<unk>"])

output_dir = "my_tokenizer/"
os.makedirs(output_dir, exist_ok=True)
tokenizer.save_model("my_tokenizer/")

['my_tokenizer/vocab.json', 'my_tokenizer/merges.txt']

In [17]:
sentence="<s> Make it shorter. </s>"
encoding = tokenizer.encode(sentence)
y=encoding.ids
print(y)

[0, 9557, 355, 13334, 17, 224, 1]


In [ ]:
if PREPROCESS:
    def convert_row(row):
        x = torch.tensor(load_relevant_data_subset(row[1].path))
        x[torch.isnan(x)] = 0
        x = feature_preprocess(x).cpu().numpy()
        sentence = row[1].sign
        return x, sentence

    def convert_and_save_data(df, limit=2043):
        total = min(df.shape[0], limit)
        npdata = np.zeros((total, 64, 80, 3))
        nplabels = np.empty(total, dtype=object)

        for i, row in tqdm(enumerate(df.iterrows()), total=total):
            if i >= limit:
                break
            x, y = convert_row(row)
            npdata[i, :, :, :] = x
            nplabels[i] = y

        np.save("feature_data.npy", npdata)
        np.save("feature_labels.npy", nplabels)

    convert_and_save_data(train_df)



100%|██████████| 20000/20000 [03:51<00:00, 86.52it/s] 


In [19]:

if PREPROCESS:
    features = np.load("/kaggle/working/feature_data.npy")
    labels = np.load("/kaggle/working/feature_labels.npy",allow_pickle=True)

print(features.shape, labels.shape)

(20000, 64, 80, 3) (20000,)


In [20]:
from torch.utils.data import Dataset, DataLoader
class PoseLandmarksDataset(Dataset):
    def __init__(self, landmarks, labels, transform=None):
        self.landmarks = torch.tensor(landmarks, dtype=torch.float32)  # Convert to tensor
        self.labels = labels  # Encoded Sentences
        self.transform = transform

    def __len__(self):
        return len(self.landmarks)

    def __getitem__(self, idx):
        sample = self.landmarks[idx]  # Shape (33, 3)
        label = self.labels[idx]
        if self.transform:
            sample = self.transform(sample)
        return sample, label
    
dataset = PoseLandmarksDataset(features, labels)

In [ ]:
class AddGaussianNoise:
    def __init__(self, mean=0.0, std=0.02):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        noise = torch.randn_like(tensor) * self.std + self.mean
        return tensor + noise

In [ ]:
train_size = int(0.95 * len(dataset))  # 80% train, 20% test
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

def collate_fn(batch):
    features, texts = zip(*batch) 
    
    
    encoded = [tokenizer.encode(str("<s>" + str(text) + "</s>")) for text in texts]
    input_ids=[e.ids for e in encoded]
    attention_masks=[e.attention_mask for e in encoded]

    max_len = max(len(ids) for ids in input_ids)
    padded_ids = [ids + [tokenizer.token_to_id("<pad>")] * (max_len - len(ids)) for ids in input_ids]
    padded_masks = [
        [1 if token != tokenizer.token_to_id("<pad>") else 0 for token in ids] 
        for ids in padded_ids
    ]

    input_ids_tensor = torch.tensor(padded_ids)
    attention_mask_tensor = torch.tensor(padded_masks)
    features_tensor = torch.stack(features)

    return {
        "input_ids": input_ids_tensor,
        "attention_mask": attention_mask_tensor,
        "features": features_tensor
    }
    
# Create DataLoaders
noise_transform = AddGaussianNoise(mean=0.0, std=0.02)
train_dataset.dataset.transform = noise_transform  # Only train set gets noise
test_dataset.dataset.transform = None             # No noise for test set
train_loader = DataLoader(train_dataset, batch_size=16, collate_fn=collate_fn, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1, collate_fn=collate_fn, shuffle=False)

In [ ]:
import numpy as np
import torch.nn as nn
import torch

class HWGATEParams():
    def __init__(self, dataset_params, input_dim, device=None) -> None:
        self.kp_dim=input_dim
        self.num_kps=80
        self.temporal_dim=dataset_params['src_len']
        self.num_classes=dataset_params['num_class']
        self.embed_dim=128
        self.temporal_patch_size=2
        self.pe=True
        self.depths=[2, 2, 4]
        self.num_heads=[4, 8, 16]
        self.window_size=16
        self.drop_rate=0.1
        self.attn_drop_rate=0.0
        self.ff_ratio=2.
        self.norm_layer=nn.LayerNorm
        self.device=device

        self.edges = [[
                    [0,1],[1,2],[3,4],[4,5],[2,6],[3,6],[6,7],[7,8],[7,9],[8,10],[10,11],[11,12],[9,13],[13,14],[14,15]

                ],
                [
                    [0,3],[0,6],[0,9],[0,12],[0,15],[1,2],[2,3],[4,5],[5,6],[7,8],[8,9],[10,11],[11,12],[13,14],[14,15]
                ],
                [
                    [0,1],[1,2],[2,3],[3,4],[4,5],[5,6],[3,7],[7,8],[3,9],[9,10],[3,11],[11,12],[3,13],[13,14],[14,15]
                ],
                [
                    [0,1],[1,2],[2,3],[3,4],[4,5],[5,6],[3,7],[7,8],[3,9],[9,10],[3,11],[11,12],[3,13],[13,14],[14,15]
                ], [
                    [0,3],[0,6],[0,9],[0,12],[0,15],[1,2],[2,3],[4,5],[5,6],[7,8],[8,9],[10,11],[11,12],[13,14],[14,15]
                ],]

        self.adj_mat = torch.tensor(self.get_adj_mat(), dtype=torch.float32)
    
    def get_adj_mat(self):
        TP, W, K = self.temporal_patch_size, self.window_size, self.num_kps
        adj_mat_adj = [self.get_adj(i) for i in range(len(self.edges))]
        adj_mat = []
        for w in range(K//W):
            adj_mat_w = []
            for i in range(TP):
                adj_mat_r = []
                for j in range(TP):
                    if i==j:
                        adj_mat_r.append(adj_mat_adj[w])
                    elif abs(i-j) == 1.0:
                        adj_mat_r.append(np.eye(W))
                    else:
                        adj_mat_r.append(np.zeros((W, W)))
                adj_mat_w.append(np.concatenate(adj_mat_r, axis=1))
            adj_mat.append(np.concatenate(adj_mat_w))
        adj_mat = np.array(adj_mat)

        return adj_mat

    def get_adj(self, index):
        temp = np.eye(self.window_size)

        for i in self.edges[index]:
            temp[tuple(i)] = 1
            temp[tuple(i)[::-1]] = 1
        return temp
    
    def get_model_params(self):
        return self.kp_dim, self.num_kps, self.temporal_dim,self.num_classes,self.embed_dim,self.temporal_patch_size,self.pe,self.depths,self.num_heads,self.window_size,self.adj_mat,self.drop_rate,self.attn_drop_rate,self.ff_ratio,self.norm_layer,self.device


In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as Fun

class KeypointPooler(nn.Module):
    def __init__(self, d_model, output_dim=1024, hidden_dim=512, dropout=0.2, use_skip=True):
        super().__init__()
        h = hidden_dim or d_model
        self.use_skip = use_skip
        
        # Multi-head attention for keypoint scoring
        self.scorer = nn.Sequential(
            nn.Linear(d_model, h),
            nn.GELU(),  # GELU often performs better than Tanh
            nn.Dropout(dropout/2),  # Lighter dropout before scoring
            nn.Linear(h, 1)
        )
        
        # Layer norm before dimension increase
        self.norm = nn.LayerNorm(d_model)
        
        # More flexible dimension increase
        self.increase_dim = nn.Sequential(
            nn.Linear(d_model, output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        
        # If using skip connection and dimensions don't match
        if use_skip and d_model != output_dim:
            self.skip_projection = nn.Linear(d_model, output_dim)
        else:
            self.skip_projection = None
            
    def forward(self, x):
        B, F, K, D = x.shape
        
        # Average pooling across keypoints
        averaged_pooled = x.mean(dim=2)  # [B, F, D]
        
        # Apply attention mechanism
        scores = self.scorer(x)  # [B, F, K, 1]
        weights = Fun.softmax(scores, dim=2)
        pooled = (weights * x).sum(dim=2)  # [B, F, D]
        
        # Apply skip connection if enabled
        if self.use_skip:
            res_pooled = pooled + averaged_pooled
        else:
            res_pooled = pooled
            
        # Apply normalization
        normalized = self.norm(res_pooled)
        
        # Increase dimensions
        output = self.increase_dim(normalized)
        
        # Apply skip connection at the output if dimensions match
        if self.use_skip and self.skip_projection is not None:
            output = output + self.skip_projection(averaged_pooled)
            
        return output

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from timm.models.layers import trunc_normal_
from torch.autograd import Variable
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Compute positional encodings
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Shape: [1, max_len, 1, d_model] for broadcasting with [B, F, K, D]
        pe = pe.unsqueeze(0).unsqueeze(2)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [B, F, K, D] -> add PE along temporal dimension F
        # pe shape: [1, max_len, 1, D] -> broadcast to [1, F, 1, D]
        x = x + self.pe[:, :x.size(1), :, :]  
        return self.dropout(x)

def window_partition(x, window_size=16, temporal_patch_size=4):
    W, TP = window_size, temporal_patch_size
    B, F, K, ED = x.shape
    f, nW = F//TP, K//W
    x = x.reshape(B, f, TP, nW, W, ED).transpose(2, 3).contiguous()  # B, F//TP, K//W, TP, W, ED
    x = x.view(B*f*nW, TP*W, ED)  # B_f_nW, W_TP, ED
    return x


def window_reverse(x, window_size=16, temporal_patch_size=4, temporal_dim=128, num_kp=64):
    W, TP = window_size, temporal_patch_size
    F, K = temporal_dim, num_kp
    B_f_nW, W_TP, ED = x.shape
    f, nW = F//TP, K//W
    B = int(B_f_nW / (f * nW))
    x = x.reshape(B, f, nW, TP, W, ED).transpose(2, 3).contiguous()  # B, F//TP, TP, K//W, W, ED
    x = x.view(B, F, K, ED)
    return x

class TemporalMerging(nn.Module):
    def __init__(self, dim, temporal_patch_size):
        super().__init__()
        self.dim = dim
        self.temporal_patch_size = temporal_patch_size
    
    def forward(self, x):
        TP = self.temporal_patch_size
        B, F, K, ED = x.shape
        f = F//TP

        x = x.reshape(B, f, TP, K, ED).transpose(2, 3).contiguous()
        x = x.reshape(B, f, K, -1).contiguous() # B F//TP ED*TP

        return x
    
class MSA(nn.Module):
    def __init__(self, dim, num_heads, adj_mat=None,
            attn_drop=0., proj_drop=0.) -> None:
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads

        assert dim%num_heads == 0, 'dim and number of heads are incompatible'

        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.adj_mat = adj_mat
        self.qkv = nn.Linear(dim, dim * 3)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, x, B, f, nW, mask=None):
        B_f_nW, W_TP, ED = x.shape
        qkv = self.qkv(x).reshape(B_f_nW, W_TP, 3, self.num_heads, ED // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = q * self.scale

        attn = (q @ k.transpose(-2, -1))

        # making some attn 0 to prevent overfitting
        if self.training:
            attn_copy = attn.detach().clone()
            seed_ = torch.rand(1).item()
            attn_copy = self.softmax(attn_copy)
            index_array = attn_copy > seed_
            index_array = torch.where(index_array == True, 0, 1)
            attn = attn * index_array

        if mask is not None:
            attn = attn.view(B, f*nW, *attn.shape[1:]) * mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(B*f*nW, *attn.shape[2:])
        
        if self.adj_mat is not None:
            adj_mat = self.adj_mat.to(x.device)
            attn = attn.view(B, f * nW, *attn.shape[1:]) * adj_mat.unsqueeze(1)

            attn = attn.view(B*f*nW, *attn.shape[2:])
        
        attn = attn.masked_fill(attn == 0, float(-10000))
        attn = self.softmax(attn)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B_f_nW, W_TP, ED)
        x = self.proj(x)
        x = self.proj_drop(x)
        # print('MSA out', x.shape)
        return x
    
class FeedForward(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x
    
class PartAttentionBlock(nn.Module):
    def __init__(self, dim, num_kps=64,
                num_heads=4, window_size=16,
                temporal_patch_size=4,
                temporal_dim=128,
                shift_size=0,
                adj_mat=None,
                drop=0., attn_drop=0.,
                ff_ratio=4.,
                act_layer=nn.GELU,
                norm_layer=nn.LayerNorm):
        super().__init__()
        self.dim = dim
        self.num_kps = num_kps
        self.num_heads = num_heads
        self.window_size = window_size
        self.temporal_patch_size = temporal_patch_size
        self.temporal_dim = temporal_dim
        self.shift_size = shift_size
        self.drop = drop
        self.attn_drop = attn_drop
        self.ff_dim = dim * ff_ratio
        self.act_layer = act_layer

        self.norm1 = norm_layer(dim)
        self.attn = MSA(dim, num_heads=num_heads, adj_mat=adj_mat,
            attn_drop=attn_drop, proj_drop=drop)

        self.norm2 = norm_layer(dim)
        self.ff = FeedForward(in_features=dim, hidden_features=int(self.ff_dim), act_layer=act_layer, drop=drop)

        if self.shift_size > 0:
            F, K = temporal_dim, num_kps
            frame_mask = torch.zeros((1, F, K, 1))  # 1 H W 1
            t_slices = (slice(0, -temporal_patch_size),
                        slice(-temporal_patch_size, -shift_size),
                        slice(-shift_size, None))

            cnt = 0
            for t in t_slices:
                frame_mask[:, t, :] = cnt
                cnt += 1

            mask_windows = window_partition(frame_mask, window_size, temporal_patch_size).squeeze(2)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(attn_mask != 0, float(0.0)).masked_fill(attn_mask == 0, float(1))
        else:
            attn_mask = None

        self.register_buffer("attn_mask", attn_mask)
    
    def forward(self, x):
        W, TP = self.window_size, self.temporal_patch_size
        B, F, K, ED = x.shape
        f, nW = F//TP, K//W

        shortcut = x

        # cyclic shift
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=-self.shift_size, dims=1)
        else:
            shifted_x = x
        x = window_partition(shifted_x, W, TP)

        x = self.norm1(x)
        # MSA/TS-MSA
        x = self.attn(x, B, f, nW, mask=self.attn_mask)  # B_f_nW, W_TP, ED

        x = window_reverse(x, W, TP, F, K)

        # cyclic shift
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=self.shift_size, dims=1)
            # partition windows
        else:
            shifted_x = x
            # partition windows

        x = shortcut + shifted_x
        # FFN
        x = x + self.ff(self.norm2(x))

        return x

class PartAttentionLayer(nn.Module):
    def __init__(self, dim, temporal_patch_size, temporal_dim, num_kps, depth, num_heads, window_size, adj_mat,
                 drop=0., attn_drop=0., ff_ratio=4., norm_layer=nn.LayerNorm, downsample=None, i_layer=0, device=None):
        super().__init__()
        self.dim = dim
        self.depth = depth
        self.num_heads = num_heads
        self.window_size = window_size
        self.adj_mat = adj_mat.to(device)
        self.i_layer = i_layer

        # build blocks
        self.blocks = nn.ModuleList([
            PartAttentionBlock(dim=dim, num_kps=num_kps,
                                 num_heads=num_heads, window_size=window_size,
                                 temporal_patch_size=temporal_patch_size,
                                 temporal_dim = temporal_dim,
                                 shift_size=0 if (i % 2 == 0) else temporal_patch_size // 2,
                                 adj_mat=self.adj_mat,
                                 drop=drop, attn_drop=attn_drop,
                                 ff_ratio=ff_ratio,
                                 norm_layer=norm_layer)
            for i in range(depth)])

        # temporal merging layer
        if downsample is not None:
            self.downsample = downsample(dim, temporal_patch_size)
        else:
            self.downsample = None

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        if self.downsample is not None:
            x = self.downsample(x)
        return x
    
class Encoder(nn.Module):
    def __init__(self,
                 kp_dim=26,
                 num_kps=64,
                 temporal_dim=256,
                 num_classes=1000,
                 embed_dim=64,
                 temporal_patch_size=4,
                 pe=False,
                 depths=[2, 2, 6, 2],
                 num_heads=[2, 4, 8, 16],
                 window_size=16,
                 adj_mat=None,
                 drop_rate=0., attn_drop_rate=0., ff_ratio=4.,
                 norm_layer=nn.LayerNorm,
                 device=None,
                 ) -> None:
        super().__init__()
        self.kp_dim = kp_dim
        self.num_kps = num_kps
        self.temporal_dim = temporal_dim
        self.window_size = window_size
        self.num_classes = num_classes
        self.num_layers = len(depths)
        self.pe = pe
        self.adj_mat = adj_mat
        self.embed_dim = embed_dim
        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))
        self.temporal_out_dim = temporal_dim // temporal_patch_size ** (self.num_layers - 1)
    
        assert self.num_kps%window_size == 0, "window size and number of kps are incompatible"
        assert self.temporal_dim%temporal_patch_size == 0, "temporal dimension and temporal patch size are incompatible"

        mapping_size = embed_dim//2
        scale = 10
        
        # fourier mapping
        B_gauss = torch.normal(0.0, 1.0, (mapping_size, self.kp_dim))
        B = B_gauss * scale
        self.B = nn.Parameter(B, requires_grad=False)

        # position encoding
        if self.pe:
            self.pos_encoder = PositionalEncoding(embed_dim, drop_rate, temporal_dim)

        # build layers
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            if adj_mat is not None:
                adj_mat_t = torch.concatenate([adj_mat for _ in range(temporal_dim // temporal_patch_size ** (i_layer+1))])
            else:
                adj_mat_t = None
            layer = PartAttentionLayer(dim=int(embed_dim * 2 ** i_layer),
                               temporal_patch_size=temporal_patch_size,
                               temporal_dim=temporal_dim//(temporal_patch_size ** i_layer),
                               num_kps=num_kps,
                               depth=depths[i_layer],
                               num_heads=num_heads[i_layer],
                               window_size=window_size,
                               adj_mat=adj_mat_t,
                               drop=drop_rate, attn_drop=attn_drop_rate, ff_ratio=ff_ratio,
                               norm_layer=norm_layer,
                               downsample=TemporalMerging if (i_layer < self.num_layers - 1) else None,
                               i_layer=i_layer,
                               device=device)
            self.layers.append(layer)

        self.norm = norm_layer(self.num_features)
        self.avgpool = nn.AvgPool1d(self.temporal_out_dim * self.num_kps)
        self.head = nn.Linear(self.num_features, num_classes) if num_classes > 0 else nn.Identity()
        self.scored=KeypointPooler(d_model=self.num_features,hidden_dim=1024)

        self.apply(self._init_weights)
        

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward_features(self, x):
        self.B = self.B.to(x.device)
        x_proj = (2.*torch.pi*x) @ self.B.transpose(1,0)
        x_ = torch.cat([torch.sin(x_proj), torch.cos(x_proj)], axis=-1)
        x = x_
        if self.pe:
            x = self.pos_encoder(x)

        for layer in self.layers:
            x = layer(x)

        B, f, K, d = x.shape
        x = self.norm(x)
        x = self.scored(x)
        return x

    def forward(self, x):
        x = self.forward_features(x)
        return x

/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [25]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dataset_params = {
    'src_len': 64, 
    'num_class': 43608# Example value, modify as per your dataset
}
model_params = HWGATEParams(dataset_params, input_dim=3, device=device)


In [26]:
encoder = Encoder(
    kp_dim=model_params.kp_dim,
    num_kps=model_params.num_kps,
    temporal_dim=model_params.temporal_dim,
    num_classes=model_params.num_classes,
    embed_dim=model_params.embed_dim,
    temporal_patch_size=model_params.temporal_patch_size,
    pe=model_params.pe,
    depths=model_params.depths,
    num_heads=model_params.num_heads,
    window_size=model_params.window_size,
    adj_mat=model_params.adj_mat,
    drop_rate=model_params.drop_rate,
    attn_drop_rate=model_params.attn_drop_rate,
    ff_ratio=model_params.ff_ratio,
    norm_layer=model_params.norm_layer,
    device=device
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim=1024, n_heads=8):
        super().__init__()
        assert embed_dim % n_heads == 0, "embed_dim must be divisible by n_heads"

        self.embed_dim = embed_dim
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads

        # Mistake fixed: project full embedding dimension, not per-head dim
        self.query_matrix = nn.Linear(embed_dim, embed_dim, bias=False)
        self.key_matrix   = nn.Linear(embed_dim, embed_dim, bias=False)
        self.value_matrix = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj     = nn.Linear(embed_dim, embed_dim)


    def forward(self, key, query, value, mask=None,padding_mask=None):
        B, T_k, _ = key.size()
        T_q = query.size(1)

        # 1) Linear projections on full embeddings
        q = self.query_matrix(query)  # [B, T_q, E]
        k = self.key_matrix(key)      # [B, T_k, E]
        v = self.value_matrix(value)  # [B, T_k, E]

        # 2) Split into heads and transpose
        def split_heads(x, T):
            return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q = split_heads(q, T_q)  # [B, H, T_q, D_h]
        k = split_heads(k, T_k)  # [B, H, T_k, D_h]
        v = split_heads(v, T_k)  # [B, H, T_k, D_h]
        # 3) Scaled dot-product attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.to(q.device)

        # 4) Apply masks safely (broadcast to [B, H, T_q, T_k])
        if (mask is not None) or (padding_mask is not None):
            attn_mask = None
            if mask is not None:
                attn_mask = mask.to(scores.device).bool()
            if padding_mask is not None:
                pm = padding_mask.to(scores.device).bool()
                attn_mask = pm if attn_mask is None else (attn_mask & pm)
            # invalid positions -> -inf
            scores = scores.masked_fill(~attn_mask, float('-inf'))
            # avoid rows of all -inf that lead to NaNs after softmax
            all_inf = torch.isneginf(scores).all(dim=-1, keepdim=True)
            scores = torch.where(all_inf, torch.zeros_like(scores), scores)
        else:
            all_inf = None

        # 5) Softmax
        attn = F.softmax(scores, dim=-1)
        # If a row had all -inf, set attention to zeros (no contribution)
        if all_inf is not None:
            attn = torch.where(all_inf, torch.zeros_like(attn), attn)

        # 6) Compute attention output
        out = torch.matmul(attn, v)  # [B, H, T_q, D_h]
        
        out = out.transpose(1, 2).contiguous().view(B, T_q, self.embed_dim)

        # 7) Final linear projection
        return self.out_proj(out)



In [ ]:
import torch.nn as nn
class Embedding(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super(Embedding, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
    def forward(self, x):
        
        out = self.embed(x)
        return out

class PositionalEmbedding(nn.Module):
    def __init__(self,max_seq_len,embed_model_dim):
        super(PositionalEmbedding, self).__init__()
        self.embed_dim = embed_model_dim

        pe = torch.zeros(max_seq_len,self.embed_dim)
        for pos in range(max_seq_len):
            for i in range(0,self.embed_dim,2):
                pe[pos, i] = math.sin(pos / (10000 ** ((2 * i)/self.embed_dim)))
                pe[pos, i + 1] = math.cos(pos / (10000 ** ((2 * (i + 1))/self.embed_dim)))
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)


    def forward(self, x):
        """
        Args:
            x: input vector
        Returns:
            x: output
        """
      
        # make embeddings relatively larger
        x = x * math.sqrt(self.embed_dim)
        #add constant to embedding
        seq_len = x.size(1)
        x = x + torch.autograd.Variable(self.pe[:, :seq_len], requires_grad=False).to(x.device)

        return x



class TransformerBlock(nn.Module):
    def __init__(self,embed_dim,context_length,expansion_factor=4,n_heads=8):
        super(TransformerBlock, self).__init__()
        self.attention=MultiHeadAttention(embed_dim,n_heads)
        self.norm1=nn.LayerNorm(embed_dim)
        self.norm2=nn.LayerNorm(embed_dim)

        self.feed_forward = nn.Sequential(
                          nn.Linear(embed_dim, expansion_factor*embed_dim),
                          nn.ReLU(),
                          nn.Linear(expansion_factor*embed_dim, embed_dim)
        )
        self.dropout1 = nn.Dropout(0.2)
        self.dropout2 = nn.Dropout(0.2)

    def forward(self,key,query,value):
        attention_out=self.attention(key,query,value)
        attention_residual_out=attention_out+query
        norm1_out=self.dropout1(self.norm1(attention_residual_out))

        feed_fwd_out = self.feed_forward(norm1_out) #32x10x512 -> #32x10x2048 -> 32x10x512
        feed_fwd_residual_out = feed_fwd_out + norm1_out #32x10x512
        norm2_out = self.dropout2(self.norm2(feed_fwd_residual_out)) #32x10x512

        return norm2_out



class DecoderBlock(nn.Module):
    def __init__(self,embed_dim,expansion_factor=4,n_heads=8):
        super(DecoderBlock,self).__init__()

        self.attention=MultiHeadAttention(embed_dim,n_heads)
        self.norm=nn.LayerNorm(embed_dim)
        self.dropout=nn.Dropout(0.2)
        self.transformer_block=TransformerBlock(embed_dim, expansion_factor, n_heads)

    def forward(self, key, query, x,mask, padding_mask):
        attention = self.attention(query,query,query,mask=mask,padding_mask=padding_mask)
        value = self.dropout(self.norm(attention + query))
        
        out = self.transformer_block(key, value, x)

        
        return out

class Decoder(nn.Module):
    def __init__(self, target_vocab_size, embed_dim, num_layers=6, expansion_factor=4, n_heads=8):
        super(Decoder, self).__init__()
        
        self.word_embedding = nn.Embedding(target_vocab_size, embed_dim)

        self.layers = nn.ModuleList(
            [
                DecoderBlock(embed_dim, expansion_factor=4, n_heads=8) 
                for _ in range(num_layers)
            ]

        )
        self.fc_out = nn.Linear(embed_dim, target_vocab_size)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x, enc_out,mask,padding_mask=None):
        seq_len = x.size(1)
        x = self.word_embedding(x)  #32x10x512
        position_embedding = PositionalEmbedding(seq_len, x.size(-1))
        x = position_embedding(x) #32x10x512
        x = self.dropout(x)
     
        for layer in self.layers:
            x = layer(enc_out, x, enc_out,mask,padding_mask) 

        #out = F.softmax(self.fc_out(x))
        out = self.fc_out(x)
        return out



In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dataset_params = {
    'src_len': 64, 
    'num_class': 43609# Example value, modify as per your dataset
}
model_params = HWGATEParams(dataset_params, input_dim=3, device=device)

# Fix 1: Corrected Signformer class with proper device handling
class CorrectedSignformer(nn.Module):
    def __init__(self, model_params, embed_dim, target_vocab_size, num_layers=2, expansion_factor=4, n_heads=8):
        super(CorrectedSignformer,self).__init__()

        self.encoder = Encoder(
                        kp_dim=model_params.kp_dim,
                        num_kps=model_params.num_kps,
                        temporal_dim=model_params.temporal_dim,
                        num_classes=model_params.num_classes,
                        embed_dim=model_params.embed_dim,
                        temporal_patch_size=model_params.temporal_patch_size,
                        pe=model_params.pe,
                        depths=model_params.depths,
                        num_heads=model_params.num_heads,
                        window_size=model_params.window_size,
                        adj_mat=model_params.adj_mat,
                        drop_rate=model_params.drop_rate,
                        attn_drop_rate=model_params.attn_drop_rate,
                        ff_ratio=model_params.ff_ratio,
                        norm_layer=model_params.norm_layer,
                        device=device
                    )
        self.decoder=Decoder(target_vocab_size,embed_dim=1024, num_layers=num_layers, expansion_factor=expansion_factor, n_heads=n_heads)

    def make_trg_mask(self, trg):
        batch_size, trg_len = trg.shape
        device = trg.device
        # returns the lower triangular part of matrix filled with ones
        trg_mask = torch.tril(torch.ones((trg_len, trg_len), device=device)).expand(
            batch_size, 1, trg_len, trg_len
        )
        return trg_mask   

    def decode(self, inputs, tokenizer, max_seq_len=81, tau=1.0):
        device = inputs.device
        enc_out = self.encoder(inputs)
        B = inputs.shape[0]

        bos_id = tokenizer.token_to_id("<s>")
        eos_id = tokenizer.token_to_id("</s>")
        pad_id = tokenizer.token_to_id("<pad>") if hasattr(tokenizer, "token_to_id") else None

        out = torch.full((B, 1), bos_id, dtype=torch.long, device=device)
        outputs = []
        finished = torch.zeros(B, dtype=torch.bool, device=device)

        for _ in range(max_seq_len):
            trg_mask = self.make_trg_mask(out)
            logits = self.decoder(out, enc_out, trg_mask)  # (B, T, V)
            step_logits = logits[:, -1, :]
            if pad_id is not None:
                step_logits[:, pad_id] = -1e9
            step_logits[:, bos_id] = -1e9
            step_logits = torch.nan_to_num(step_logits, neginf=-1e9, posinf=1e9)
            step_logits = torch.clamp(step_logits, -50, 50)

            if tau == 1.0:
                pred_token = step_logits.argmax(dim=-1)
            else:
                probs = F.softmax(step_logits / max(1e-3, tau), dim=-1)
                pred_token = torch.multinomial(probs, 1).squeeze(-1)

            pred_token = torch.where(finished, torch.full_like(pred_token, eos_id), pred_token)
            outputs.append(step_logits.unsqueeze(1))
            out = torch.cat([out, pred_token.unsqueeze(1)], dim=1)
            finished |= (pred_token == eos_id)
            if finished.all():
                break

        return torch.cat(outputs, dim=1)

    def decode_without(self, inputs, tokenizer, max_seq_len=81, tau=1.0, beam_size=1):
        device = inputs.device
        enc_out = self.encoder(inputs)
        B = inputs.shape[0]
        
        # Handle beam search vs greedy decoding
        if beam_size == 1:
            # Original greedy decoding logic
            bos_id = tokenizer.token_to_id("<s>")
            pad_id = tokenizer.token_to_id("<pad>") if hasattr(tokenizer, "token_to_id") else None

            out = torch.full((B, 1), bos_id, dtype=torch.long, device=device)
            outputs = []

            for _ in range(max_seq_len):
                trg_mask = self.make_trg_mask(out)
                logits = self.decoder(out, enc_out, trg_mask)
                step_logits = logits[:, -1, :]
                if pad_id is not None:
                    step_logits[:, pad_id] = -1e9
                step_logits[:, bos_id] = -1e9
                bad = ~torch.isfinite(step_logits)
                if bad.any():
                    rows_all_bad = bad.all(dim=-1)
                    if rows_all_bad.any():
                        step_logits[rows_all_bad] = 0.0
                step_logits = torch.nan_to_num(step_logits, neginf=-1e9, posinf=1e9)
                step_logits = torch.clamp(step_logits, -50, 50)

                outputs.append(step_logits.unsqueeze(1))
                soft_sample = F.gumbel_softmax(step_logits, tau=max(1e-3, tau), hard=True)
                next_token = soft_sample.argmax(dim=-1)
                out = torch.cat([out, next_token.unsqueeze(1)], dim=1)

            return torch.cat(outputs, dim=1)
        
        else:
            # Beam search implementation
            bos_id = tokenizer.token_to_id("<s>")
            eos_id = tokenizer.token_to_id("</s>")
            pad_id = tokenizer.token_to_id("<pad>") if hasattr(tokenizer, "token_to_id") else None
            
            # Initialize beam states
            beam_tokens = torch.full((B * beam_size, 1), bos_id, dtype=torch.long, device=device)
            beam_scores = torch.zeros(B * beam_size, device=device)
            beam_finished = torch.zeros(B * beam_size, dtype=torch.bool, device=device)
            
            # Expand encoder output for beam search
            enc_out = enc_out.unsqueeze(1).expand(-1, beam_size, -1, -1).contiguous().view(B * beam_size, -1, -1)
            
            outputs = []
            
            for step in range(max_seq_len):
                trg_mask = self.make_trg_mask(beam_tokens)
                logits = self.decoder(beam_tokens, enc_out, trg_mask)
                step_logits = logits[:, -1, :]  # (B*beam_size, vocab_size)
                
                # Apply token filtering
                if pad_id is not None:
                    step_logits[:, pad_id] = -1e9
                step_logits[:, bos_id] = -1e9
                
                # Handle numerical stability
                bad = ~torch.isfinite(step_logits)
                if bad.any():
                    rows_all_bad = bad.all(dim=-1)
                    if rows_all_bad.any():
                        step_logits[rows_all_bad] = 0.0
                step_logits = torch.nan_to_num(step_logits, neginf=-1e9, posinf=1e9)
                step_logits = torch.clamp(step_logits, -50, 50)
                
                # Get top-k candidates for each beam
                vocab_size = step_logits.size(-1)
                step_logits = step_logits.view(B, beam_size, vocab_size)
                
                # Add current beam scores
                step_scores = step_logits + beam_scores.view(B, beam_size, 1)
                
                # Get top-k candidates
                top_scores, top_indices = torch.topk(step_scores.view(B, -1), beam_size, dim=-1)
                top_scores = top_scores.view(B, beam_size)
                top_indices = top_indices.view(B, beam_size)
                
                # Convert flat indices back to beam and token indices
                beam_indices = top_indices // vocab_size
                token_indices = top_indices % vocab_size
                
                # Update beam states
                new_beam_tokens = []
                new_beam_scores = []
                new_beam_finished = []
                
                for b in range(B):
                    for k in range(beam_size):
                        beam_idx = beam_indices[b, k]
                        token_idx = token_indices[b, k]
                        
                        # Get the previous sequence for this beam
                        prev_seq = beam_tokens[b * beam_size + beam_idx]
                        new_seq = torch.cat([prev_seq, token_idx.unsqueeze(0)], dim=0)
                        new_beam_tokens.append(new_seq)
                        new_beam_scores.append(top_scores[b, k])
                        new_beam_finished.append(beam_finished[b * beam_size + beam_idx] | (token_idx == eos_id))
                
                # Update beam states
                beam_tokens = torch.stack(new_beam_tokens)
                beam_scores = torch.stack(new_beam_scores)
                beam_finished = torch.stack(new_beam_finished)
                
                # Store outputs (use the best beam for each batch)
                best_beams = beam_scores.view(B, beam_size).argmax(dim=-1)
                best_outputs = []
                for b in range(B):
                    best_idx = b * beam_size + best_beams[b]
                    best_outputs.append(step_logits[b, best_beams[b]])
                outputs.append(torch.stack(best_outputs).unsqueeze(1))
                
                # Check if all beams are finished
                if beam_finished.all():
                    break
            
            return torch.cat(outputs, dim=1)

    def forward_without(self, inputs, labels, padding_mask):
        trg_mask = self.make_trg_mask(labels)  # Already on correct device
        enc_out = self.encoder(inputs)
        outputs = self.decoder(labels, enc_out, trg_mask, padding_mask)
        return outputs

    def forward(
        self, 
        inputs, 
        labels, 
        padding_mask, 
        tokenizer, 
        use_decode_prob=0.0
    ):
        """
        Forward pass that probabilistically chooses between:
        - forward() with teacher forcing
        - decode() with autoregressive prediction
    
        Args:
            inputs:        (B, src_len, feat_dim)
            labels:        (B, T)
            padding_mask:  optional
            tokenizer:     tokenizer with <s> and </s> tokens
            use_decode_prob: float in [0, 1], probability of using decode() (i.e., autoregressive)
        Returns:
            logits: (B, T, V)
        """
        if torch.rand(1).item() < use_decode_prob:
            logits = self.decode_without(inputs, tokenizer, max_seq_len=labels.shape[1],tau=1.0, beam_size=4)
        else:
            # Full teacher forcing mode
            logits = self.forward_without(inputs, labels, padding_mask)
    
        return logits

In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MaskedCrossEntropyLoss(nn.Module):
    """
    Enhanced loss function specifically designed for sign language translation tasks.
    Combines label smoothing with better handling of sequence generation and class imbalance.
    """
    def __init__(self, label_smoothing=0.1, length_penalty=0.05, ignore_index=-100):
        super().__init__()
        self.label_smoothing = label_smoothing
        self.length_penalty = length_penalty
        self.ignore_index = ignore_index

    def forward(self, logits: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # logits: (B, T, V), target: (B, T), mask: (B, T)
        B, T, V = logits.size()
        
        # Apply label smoothing for better generalization
        if self.label_smoothing > 0:
            # Create smoothed targets
            target_one_hot = F.one_hot(target, num_classes=V).float()
            target_one_hot = target_one_hot * (1 - self.label_smoothing) + self.label_smoothing / V
            
            # Compute KL divergence loss (more stable than cross-entropy with smoothing)
            log_probs = F.log_softmax(logits, dim=-1)
            loss_per_token = -(target_one_hot * log_probs).sum(dim=-1)
        else:
            # Standard cross entropy
            loss_per_token = F.cross_entropy(
                logits.view(-1, V),
                target.view(-1),
                reduction='none',
                ignore_index=self.ignore_index
            ).view(B, T)

        # Apply mask to ignore padding tokens
        masked_loss = loss_per_token * mask
        
        # Add length penalty to encourage more concise translations
        if self.length_penalty > 0:
            seq_lengths = mask.sum(dim=1).float()
            avg_length = seq_lengths.mean()
            # Normalize by target length to avoid bias
            target_lengths = mask.sum(dim=1).float()
            normalized_length = avg_length / target_lengths.mean().clamp(min=1)
            length_penalty_loss = self.length_penalty * normalized_length
        else:
            length_penalty_loss = 0.0
        
        # Compute final loss
        main_loss = masked_loss.sum() / mask.sum().clamp(min=1)
        total_loss = main_loss + length_penalty_loss
        
        return total_loss


In [31]:
print(tokenizer.token_to_id("<pad>"))
pad_token_id=tokenizer.token_to_id("<pad>")

2


In [32]:
import torch

def compute_accuracy(logits: torch.Tensor,
                     target: torch.Tensor,
                     mask: torch.Tensor) -> float:
    """
    Computes token accuracy for seq2seq outputs, using the provided mask.

    Args:
        logits: Tensor of shape (B, T, V) — raw model scores
        target: Tensor of shape (B, T)    — true token IDs
        mask: Tensor of shape (B, T)      — 1 for valid tokens, 0 for pad tokens

    Returns:
        accuracy: float                   — (# correct valid tokens) / (# valid tokens)
    """
    # 1) Get predicted token IDs
    preds = logits.argmax(dim=-1)  # (B, T)

    # 2) Use the given mask (expected to be 1 for valid positions, 0 for pads)
    mask = mask.to(dtype=torch.bool)

    # 3) Count correct predictions where mask is True
    correct = (preds == target) & mask
    correct_tokens = correct.sum().item()

    # 4) Count total valid tokens
    total_tokens = mask.sum().item()
    if total_tokens == 0:
        return 0.0

    return correct_tokens / total_tokens


In [33]:
print(

SyntaxError: incomplete input (149104261.py, line 1)

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

_smooth = SmoothingFunction().method4

def _safe_decode_ids(ids, tokenizer):
    try:
        if hasattr(tokenizer, 'get_vocab_size'):
            V = tokenizer.get_vocab_size()
            ids = [int(t) for t in ids if 0 <= int(t) < V]
        return tokenizer.decode(ids)
    except Exception:
        return ""

def compute_bleu(preds, targets, tokenizer):
    scores = []
    for pred, target in zip(preds, targets):
        pred_text = _safe_decode_ids(pred.tolist(), tokenizer)
        target_text = _safe_decode_ids(target.tolist(), tokenizer)
        pred_tokens = pred_text.split()
        target_tokens = target_text.split()
        if len(pred_tokens) == 0 or len(target_tokens) == 0:
            scores.append(0.0)
        else:
            scores.append(sentence_bleu([target_tokens], pred_tokens, smoothing_function=_smooth))
    return sum(scores) / max(1, len(scores))

In [ ]:
def improved_scheduled_sampling(epoch, total_epochs=200, start_epoch=60, max_prob=0.2, steepness=3):
    """
    Scheduled sampling starts after 'start_epoch' with a gentle cap of 0.2
    """
    if epoch < start_epoch:
        return 0.0  # Pure teacher forcing during warmup
    x = (epoch - start_epoch) / max(1, (total_epochs - start_epoch))
    x = torch.clamp(torch.tensor(x), 0, 1)
    return float(max_prob / (1 + torch.exp(-steepness * (x - 0.5))))

In [ ]:
import shutil

shutil.copy("/kaggle/input/checkpoint-train/best_train_epoch.pt", "/kaggle/working/checkpoints_train/best_signformer_epoch11.pt")


In [36]:
import os
import torch
import torch.optim as optim
import torch.nn as nn


# Determine device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def save_checkpoint(model, optimizer, epoch,
                    train_loss=None, train_acc=None,
                    val_loss=None,   val_acc=None,
                    path=None):
    """
    Saves model and optimizer states along with training and validation metrics.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
    if train_loss is not None: payload["train_loss"] = train_loss
    if train_acc  is not None: payload["train_acc"]  = train_acc
    if val_loss   is not None: payload["val_loss"]   = val_loss
    if val_acc    is not None: payload["val_acc"]    = val_acc

    torch.save(payload, path)
    print(f"Checkpoint saved to {path}")

def load_checkpoint(model, optimizer, path, device=device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt.get("epoch", 0) + 1
    best_train_loss = ckpt.get("train_loss", float('inf'))
    print(f"Loaded checkpoint '{path}' (epoch {ckpt['epoch']}, train_loss={best_train_loss:.4f})")
    return start_epoch, best_train_loss


# def load_checkpoint(model, optimizer, path, device=device):
#     """
#     Loads checkpoint and restores model & optimizer states.
#     Returns start_epoch, best_train_loss, best_val_loss
#     """
#     if not os.path.isfile(path):
#         raise FileNotFoundError(f"No checkpoint found at '{path}'")
#     ckpt = torch.load(path, map_location=device)
#     model.load_state_dict(ckpt["model_state_dict"])
#     optimizer.load_state_dict(ckpt["optimizer_state_dict"])
#     start_epoch = ckpt.get("epoch", 0) + 1
#     best_train_loss = ckpt.get("train_loss", float('inf'))
#     best_val_loss   = ckpt.get("val_loss",   float('inf'))
#     print(f"Loaded checkpoint '{path}' (epoch {ckpt['epoch']}, train_loss={best_train_loss:.4f}, val_loss={best_val_loss:.4f})")
#     return start_epoch, best_train_loss, best_val_loss


# Fix 3: Improved evaluation function
def improved_evaluate_model(model, loader, criterion, tokenizer, k=5):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    total_bleu = 0.0
    batches = 0
    
    with torch.no_grad():
        for batch in loader:
            inputs = batch["features"].to(device)
            labels = batch["input_ids"].to(device)
            padding_mask = batch["attention_mask"]
            
            # Generate autoregressively
            logits = model.decode(inputs, tokenizer, max_seq_len=labels.shape[1])
            
            # Align sequences properly
            B, T_gen, V = logits.shape
            target_labels = labels[:, 1:]  # Remove BOS token
            
            # Use minimum length to avoid dimension mismatch
            min_len = min(T_gen, target_labels.size(1))
            logits = logits[:, :min_len]
            target_labels = target_labels[:, :min_len]
            
            # Create proper mask (ignore padding tokens)
            mask = (target_labels != tokenizer.token_to_id("<pad>")).float()
            
            # Compute loss and accuracy
            loss = criterion(logits, target_labels, mask)
            acc = compute_accuracy(logits, target_labels, mask)

            # BLEU calculation
            preds = logits.argmax(dim=-1)
            bleu_batch = 0.0
            for pred, target in zip(preds, target_labels):
                pred_tokens = tokenizer.decode(pred.tolist()).split()
                target_tokens = tokenizer.decode(target.tolist()).split()
                bleu_batch += sentence_bleu([target_tokens], pred_tokens)
            bleu_batch /= preds.size(0)
            total_bleu += bleu_batch
            
            total_loss += loss.item()
            total_acc += acc
            batches += 1

    avg_bleu = total_bleu / batches if batches else 0.0
    return total_loss / batches, total_acc / batches, avg_bleu

def evaluate_model_with_teacher_forcing(model, loader, criterion, tokenizer, k=5):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    batches = 0
    # curr_prob = delayed_sigmoid_schedule(epoch, 50, warmup_ratio=0.2, max_prob=0.5)
    with torch.no_grad():
        for batch in loader:
            inputs = batch["features"].to(device)
            labels = batch["input_ids"].to(device)
            padding_mask = batch["attention_mask"]
            loss_padding_mask = padding_mask[:, 1:].to(device)
            decoder_padding_mask = padding_mask[:, :-1].unsqueeze(1).unsqueeze(2).to(device)

            decoder_input = labels[:, :-1]
            target_labels = labels[:, 1:]

            outputs = model(inputs, decoder_input, decoder_padding_mask, tokenizer, use_decode_prob=0.0)
            loss = criterion(outputs, target_labels, loss_padding_mask)
            
            total_loss += loss.item()
            total_acc += compute_accuracy(outputs, target_labels, loss_padding_mask)
            batches += 1

    avg_loss = total_loss / batches if batches else 0.0
    avg_acc = total_acc / batches if batches else 0.0
    return avg_loss, avg_acc

def delayed_sigmoid_schedule(epoch, total_epochs=50, warmup_ratio=0.3, max_prob=0.5, steepness=6):
    x = (epoch - warmup_ratio * total_epochs) / (total_epochs * (1 - warmup_ratio))
    x = torch.clamp(torch.tensor(x), 0, 1)  # Clamp to [0, 1] after warmup
    return float(max_prob / (1 + torch.exp(-steepness * (x - 0.5))))
    
# Fix 4: Improved training function with gradient clipping and early stopping
def improved_train_model(model, train_loader, val_loader, tokenizer,
                        num_epochs=50, learning_rate=5e-5, k=5,
                        ckpt_dir="checkpoints_train", resume_from=None):
    raw_model = model.to(device)
    dp_model = nn.DataParallel(raw_model)
    criterion = MaskedCrossEntropyLoss()
    optimizer = optim.AdamW(dp_model.parameters(), lr=learning_rate, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    start_epoch = 1
    best_train_loss = float('inf')
    best_val_loss = float('inf')
    patience = 200
    no_improve_count = 0
    
    # Optionally resume from checkpoint
    if resume_from:
        checkpoint_path = os.path.join(ckpt_dir, resume_from)
        start_epoch, best_train_loss = load_checkpoint(raw_model, optimizer, checkpoint_path)
        dp_model = nn.DataParallel(raw_model)

    for epoch in range(start_epoch, num_epochs+1):
        # Training phase
        dp_model.train()
        running_loss = 0.0
        running_acc = 0.0
        train_batches = 0
        
        # Get scheduled sampling probability
        curr_prob = improved_scheduled_sampling(epoch, num_epochs, 60,  max_prob=0.3)
        
        print(f"Epoch {epoch}: Scheduled sampling prob = {curr_prob:.3f}")

        for batch in train_loader:
            inputs = batch["features"].to(device)
            labels = batch["input_ids"].to(device)
            padding_mask = batch["attention_mask"]
            loss_padding_mask = padding_mask[:, 1:].to(device)
            decoder_padding_mask = padding_mask[:, :-1].unsqueeze(1).unsqueeze(2).to(device)

            decoder_input = labels[:, :-1]
            target_labels = labels[:, 1:]

            optimizer.zero_grad()
            outputs = dp_model(inputs, decoder_input, decoder_padding_mask, tokenizer, use_decode_prob=curr_prob)
            loss = criterion(outputs, target_labels, loss_padding_mask)
            loss.backward()
            
            # Add gradient clipping
            torch.nn.utils.clip_grad_norm_(dp_model.parameters(), max_norm=1.0)
            
            optimizer.step()

            running_loss += loss.item()
            running_acc += compute_accuracy(outputs, target_labels, loss_padding_mask)
            train_batches += 1

        # Update learning rate
        scheduler.step()

        avg_train_loss = running_loss / train_batches
        avg_train_acc = running_acc / train_batches
        print(f"Epoch {epoch}/{num_epochs} - Train Loss: {avg_train_loss:.4f}, Acc: {avg_train_acc:.4f}")
        
        # Teacher forcing validation
        teach_val_loss, teach_val_acc = evaluate_model_with_teacher_forcing(raw_model, val_loader, criterion, tokenizer, k)
        print(f"Epoch {epoch}/{num_epochs} - TF Val Loss: {teach_val_loss:.4f}, Acc: {teach_val_acc:.4f}")
        
        # Autoregressive validation
        val_loss, val_acc, val_bleu = improved_evaluate_model(model, val_loader, criterion, tokenizer, k)
        print(f"AR Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, BLEU: {val_bleu:.4f}")

        if avg_train_loss < best_train_loss:
            best_train_loss = avg_train_loss
            train_ckpt = os.path.join(ckpt_dir, f"best_train_epoch.pt")
            save_checkpoint(raw_model, optimizer, epoch,
                            train_loss=avg_train_loss, train_acc=avg_train_acc,
                            val_loss=None, val_acc=None,
                            path=train_ckpt)
            print(f"📈 New best training loss saved (Loss: {avg_train_loss:.4f})")
            
        # Early stopping and checkpointing
        if teach_val_loss < best_val_loss:
            best_val_loss = teach_val_loss
            no_improve_count = 0
            ckpt_path = os.path.join(ckpt_dir, f"best_val_epoch.pt")
            save_checkpoint(raw_model, optimizer, epoch, train_loss=None,train_acc=None ,val_loss=teach_val_loss, val_acc=teach_val_acc,path=ckpt_path)
            print(f"🎉 New best model saved (AR Val Loss: {teach_val_loss:.4f})")
        else:
            no_improve_count += 1
            if no_improve_count >= patience:
                print(f"Early stopping triggered after {patience} epochs without improvement")
                break

    print("Training complete.")

Using device: cuda


In [ ]:
# Replace your existing model with the corrected one
model = CorrectedSignformer(model_params=model_params, embed_dim=1024, target_vocab_size=43609)

# Start training with improved parameters
num_epochs = 200
learning_rate = 5e-5  # Reduced learning rate
improved_train_model(model, train_loader, test_loader, tokenizer, num_epochs, learning_rate)

In [ ]:
# Path to your checkpoint (update the filename as needed)
resume_checkpoint = "best_train_epoch.pt"  # or whichever epoch you want

# Directory where checkpoints are saved
ckpt_dir = "/kaggle/working/checkpoints_train/"

# Re-instantiate your model (same as before)
model = CorrectedSignformer(model_params=model_params, embed_dim=1024, target_vocab_size=43609)

# Set your training parameters
num_epochs = 200
learning_rate = 5e-5

# Resume training from the checkpoint
improved_train_model(
    model,
    train_loader,
    test_loader,
    tokenizer,
    num_epochs=num_epochs,
    learning_rate=learning_rate,
    ckpt_dir=ckpt_dir,
    resume_from=resume_checkpoint
)

Loaded checkpoint '/kaggle/working/checkpoints_train/best_train_epoch.pt' (epoch 59, train_loss=2.1356)
Epoch 60: Scheduled sampling prob = 0.036
Epoch 60/200 - Train Loss: 2.4309, Acc: 0.7966
Epoch 60/200 - TF Val Loss: 1.9107, Acc: 0.9291
AR Val Loss: 3.9629, Acc: 0.7227, BLEU: 0.3837
Checkpoint saved to /kaggle/working/checkpoints_train/best_val_epoch.pt
🎉 New best model saved (AR Val Loss: 1.9107)
Epoch 61: Scheduled sampling prob = 0.037
Epoch 61/200 - Train Loss: 2.3908, Acc: 0.8055
Epoch 61/200 - TF Val Loss: 1.9414, Acc: 0.9165
AR Val Loss: 4.2772, Acc: 0.6843, BLEU: 0.3533
Epoch 62: Scheduled sampling prob = 0.038
Epoch 62/200 - Train Loss: 2.3391, Acc: 0.8181
Epoch 62/200 - TF Val Loss: 1.9919, Acc: 0.8978
AR Val Loss: 4.6745, Acc: 0.6288, BLEU: 0.3131
Epoch 63: Scheduled sampling prob = 0.039
Epoch 63/200 - Train Loss: 2.2578, Acc: 0.8309
Epoch 63/200 - TF Val Loss: 2.0349, Acc: 0.8782
AR Val Loss: 5.1922, Acc: 0.5647, BLEU: 0.2681
Epoch 64: Scheduled sampling prob = 0.040
E

In [40]:
import os

print(os.path.exists("/kaggle/working/checkpoints_train/best_train_epoch.pt"))


True


In [41]:
from IPython.display import FileLink
from IPython.display import display

display(FileLink(r'/kaggle/working/checkpoints_train/best_train_epoch.pt'))


/kaggle/working/checkpoints_train/best_train_epoch.pt

In [42]:
!zip -r best_checkpoints.zip checkpoints_train/


  adding: checkpoints_train/ (stored 0%)
  adding: checkpoints_train/best_signformer_epoch11.pt

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 25%)
  adding: checkpoints_train/best_signformer_epoch.pt (deflated 33%)
  adding: checkpoints_train/best_train_epoch.pt (deflated 25%)
  adding: checkpoints_train/best_val_epoch.pt (deflated 24%)
  adding: checkpoints_train/best_signformer_epoch1.pt (deflated 34%)


In [43]:
display(FileLink('best_checkpoints.zip'))


/kaggle/working/best_checkpoints.zip

In [ ]:
# # --- Stabilization & evaluation utilities (append) ---
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# # Safer BLEU with smoothing and robust decoding
# _smooth = SmoothingFunction().method4

# def _safe_decode_ids(ids, tokenizer):
#     try:
#         if hasattr(tokenizer, 'get_vocab_size'):
#             V = tokenizer.get_vocab_size()
#             ids = [int(t) for t in ids if 0 <= int(t) < V]
#         return tokenizer.decode(ids)
#     except Exception:
#         return ""

# def compute_bleu(preds, targets, tokenizer):
#     scores = []
#     for pred, target in zip(preds, targets):
#         pred_text = _safe_decode_ids(pred.tolist(), tokenizer)
#         target_text = _safe_decode_ids(target.tolist(), tokenizer)
#         pred_tokens = pred_text.split()
#         target_tokens = target_text.split()
#         if len(pred_tokens) == 0 or len(target_tokens) == 0:
#             scores.append(0.0)
#         else:
#             scores.append(sentence_bleu([target_tokens], pred_tokens, smoothing_function=_smooth))
#     return sum(scores) / max(1, len(scores))

# # Patch decode paths to avoid NaNs/Inf during AR generation

# def _patched_decode(self, inputs, tokenizer, max_seq_len=81, tau=1.0):
#     device = inputs.device
#     enc_out = self.encoder(inputs)
#     B = inputs.shape[0]

#     bos_id = tokenizer.token_to_id("<s>")
#     eos_id = tokenizer.token_to_id("</s>")
#     pad_id = tokenizer.token_to_id("<pad>") if hasattr(tokenizer, "token_to_id") else None

#     out = torch.full((B, 1), bos_id, dtype=torch.long, device=device)
#     outputs = []
#     finished = torch.zeros(B, dtype=torch.bool, device=device)

#     for _ in range(max_seq_len):
#         trg_mask = self.make_trg_mask(out)
#         logits = self.decoder(out, enc_out, trg_mask)  # (B, T, V)
#         step_logits = logits[:, -1, :]
#         if pad_id is not None:
#             step_logits[:, pad_id] = -1e9
#         step_logits[:, bos_id] = -1e9
#         step_logits = torch.nan_to_num(step_logits, neginf=-1e9, posinf=1e9)
#         step_logits = torch.clamp(step_logits, -50, 50)

#         if tau == 1.0:
#             pred_token = step_logits.argmax(dim=-1)
#         else:
#             probs = F.softmax(step_logits / max(1e-3, tau), dim=-1)
#             pred_token = torch.multinomial(probs, 1).squeeze(-1)

#         pred_token = torch.where(finished, torch.full_like(pred_token, eos_id), pred_token)
#         outputs.append(step_logits.unsqueeze(1))
#         out = torch.cat([out, pred_token.unsqueeze(1)], dim=1)
#         finished |= (pred_token == eos_id)
#         if finished.all():
#             break

#     return torch.cat(outputs, dim=1)


# def _patched_decode_without(self, inputs, tokenizer, max_seq_len=81, tau=1.0):
#     device = inputs.device
#     enc_out = self.encoder(inputs)
#     B = inputs.shape[0]

#     bos_id = tokenizer.token_to_id("<s>")
#     pad_id = tokenizer.token_to_id("<pad>") if hasattr(tokenizer, "token_to_id") else None

#     out = torch.full((B, 1), bos_id, dtype=torch.long, device=device)
#     outputs = []

#     for _ in range(max_seq_len):
#         trg_mask = self.make_trg_mask(out)
#         logits = self.decoder(out, enc_out, trg_mask)
#         step_logits = logits[:, -1, :]
#         if pad_id is not None:
#             step_logits[:, pad_id] = -1e9
#         step_logits[:, bos_id] = -1e9
#         bad = ~torch.isfinite(step_logits)
#         if bad.any():
#             rows_all_bad = bad.all(dim=-1)
#             if rows_all_bad.any():
#                 step_logits[rows_all_bad] = 0.0
#         step_logits = torch.nan_to_num(step_logits, neginf=-1e9, posinf=1e9)
#         step_logits = torch.clamp(step_logits, -50, 50)

#         outputs.append(step_logits.unsqueeze(1))
#         soft_sample = F.gumbel_softmax(step_logits, tau=max(1e-3, tau), hard=True)
#         next_token = soft_sample.argmax(dim=-1)
#         out = torch.cat([out, next_token.unsqueeze(1)], dim=1)

#     return torch.cat(outputs, dim=1)

# # Apply monkey patches
# CorrectedSignformer.decode = _patched_decode
# CorrectedSignformer.decode_without = _patched_decode_without

# # Gentler scheduled sampling (caps at 0.2)

# def improved_scheduled_sampling(epoch, total_epochs=200, start_epoch=60, max_prob=0.2, steepness=3):
#     if epoch < start_epoch:
#         return 0.0
#     x = (epoch - start_epoch) / max(1, (total_epochs - start_epoch))
#     x = torch.clamp(torch.tensor(x), 0, 1)
#     return float(max_prob / (1 + torch.exp(-steepness * (x - 0.5))))

# try:
#     print("Stability helpers loaded. Vocab size:", tokenizer.get_vocab_size())
# except Exception:
#     print("Stability helpers loaded.")

